# Chapitre 4 — Raisonner avec des probabilités · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch04_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch04_corrige.ipynb`, sur la même page compagnon.

**Sommaire.** Données · Les calculs des leçons 4.1 à 4.5 · Exercices 4.1 à 4.6

**Avant chaque calcul**, demandez-vous sur quel ensemble vous comptez : des spams, des messages signalés ou des messages reçus.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLEU, ORANGE = '#2a78d6', '#eb6834'   # « aucun spam manqué », courbe de la loi normale
from math import comb

## Données

Les identifiants D4.1 et D4.2 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 4, recopiées des fichiers CSV du livre (ne pas modifier)
# Lignes : spam, légitime. Colonnes : signalé, non signalé.
LIGNES, COLONNES = ['spam', 'légitime'], ['signalé', 'non signalé']

# D4.1 — boîte où 20 % des messages sont des spams (1 000 messages)
boite_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=LIGNES, columns=COLONNES)

# D4.2 — même filtre, boîte où 2 % des messages sont des spams (1 000 messages)
boite_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=LIGNES, columns=COLONNES)

# D4.3 — 1 000 durées de traitement (ms), régénérées à l'identique du fichier CSV
durees = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 4.1 à 4.5.

In [ ]:
# Leçon 4.1 — compter, puis diviser par le bon groupe (équation 4.1)
t = boite_20
n = t.values.sum()
print('total :', n, '| spams :', t.loc['spam'].sum(), '| signalés :', t['signalé'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(signalé) =', t['signalé'].sum() / n,
      '| P(spam et signalé) =', t.loc['spam', 'signalé'] / n)
# Parmi les signalés : le dénominateur est la colonne « signalé »
print(f"spams parmi les signalés : {t.loc['spam', 'signalé']} / {t['signalé'].sum()} "
      f"≈ {t.loc['spam', 'signalé'] / t['signalé'].sum():.2f}")

In [ ]:
# Leçon 4.2 — le même filtre dans une boîte où les spams sont rares (équation 4.2)
d, f = 0.90, 0.05                   # parts signalées : des spams, des messages légitimes
for p, t in [(0.20, boite_20), (0.02, boite_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    reconstruit = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(reconstruit) == t.values).all()   # le tableau se déduit des taux
    print(f'taux de base {p:.0%} : {t.values.tolist()}  →  part de spams parmi les signalés '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Leçon 4.3 — deux probabilités conditionnelles, deux dénominateurs (équation 4.3)
def conditionnelle(t, case, groupe):
    """Part de la case (ligne, colonne) dans le groupe : une ligne ('spam') ou une colonne ('signalé')."""
    denominateur = t.loc[groupe].sum() if groupe in t.index else t[groupe].sum()
    return t.loc[case] / denominateur, denominateur

for nom, t in [('20 %', boite_20), ('2 %', boite_2)]:
    rappel, den_r = conditionnelle(t, ('spam', 'signalé'), 'spam')
    precision, den_p = conditionnelle(t, ('spam', 'signalé'), 'signalé')
    print(f'boîte à {nom} : P(signalé | spam) = {rappel:.2f} (dénominateur {den_r}) ; '
          f'P(spam | signalé) = {precision:.2f} (dénominateur {den_p})')

# Les mêmes nombres avec scikit-learn, à partir d'étiquettes message par message (s'il est installé)
try:
    from sklearn.metrics import precision_score, recall_score
    vrai = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predit = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = signalé
    print(f'scikit-learn : rappel = {recall_score(vrai, predit):.2f}, précision = {precision_score(vrai, predit):.2f}')
except ImportError:
    print('scikit-learn absent : ce calcul est facultatif.')

In [ ]:
# Leçon 4.4 — le théorème de Bayes (équations 4.4 et 4.5) et la courbe du livre
def bayes(p, d=0.90, f=0.05):
    p_signale = d * p + f * (1 - p)                  # équation 4.5
    return d * p / p_signale                        # équation 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p} : P(signalé) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | signalé) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=SIGNALE, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Taux de base : part de spams (%)'); plt.ylabel('P(spam | signalé)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Leçon 4.5 — dix spams : loi binomiale (équation 4.7) et événement complémentaire (équation 4.8)
n, p = 10, 0.1                        # on compte sur 10 SPAMS ; p = probabilité d'être manqué
loi = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| au moins un manqué =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 à 4 :', [round(v, 3) for v in loi[:5]], '| somme =', round(sum(loi), 6))
couleurs = [BLEU] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), loi, color=couleurs, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Nombre de spams manqués parmi 10 spams'); plt.ylabel('Probabilité'); plt.show()

# Aperçu de la loi normale : durées de traitement (D4.3)
m, s = durees.mean(), durees.std()
print(f'moyenne = {m:.2f} ms, écart-type = {s:.2f} ms ; à ±1 écart-type : {np.mean(abs(durees - m) <= s):.1%}, '
      f'à ±2 écarts-types : {np.mean(abs(durees - m) <= 2 * s):.1%}')

## Exercice 4.1 — Lire un graphique : vingt spams

*Notions : loi binomiale, événement complémentaire (leçon 4.5).* Lisez d'abord la figure du livre.

### a) à d) Vos lectures

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : la probabilité binomiale P(X = k) (équation 4.7)
def binomiale(n, p, k):
    return None   # À COMPLÉTER ; comb(n, k) donne le coefficient binomial

In [ ]:
# Vérification sur les 20 spams de l'exercice
if binomiale(20, 0.05, 0) is None:
    print('Complétez la fonction.')
else:
    p0, p1 = binomiale(20, 0.05, 0), binomiale(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, au moins un = {1 - p0:.4f} →',
          'correct' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'à revoir')

## Exercice 4.2 — Un filtre plus prudent

*Notions : probabilité conditionnelle (leçon 4.3), théorème de Bayes (leçon 4.4), taux de base (leçon 4.2).* Le filtre prudent signale 75 % des spams et 1 % des messages légitimes. Boîte de 10 000 messages, dont 10 % de spams. Faites d'abord les questions a) à e) sur papier.

### a) à c) Sur papier

*Vos réponses :* …

### a) Le tableau en Python

In [ ]:
# À COMPLÉTER : les quatre effectifs du tableau, pour 10 000 messages
spam_signale = None
spam_non_signale = None
legitime_signale = None
legitime_non_signale = None

In [ ]:
# Vérification
cases = [spam_signale, spam_non_signale, legitime_signale, legitime_non_signale]
if None in cases:
    print('Complétez la cellule précédente.')
else:
    prudent = pd.DataFrame([cases[:2], cases[2:]], index=LIGNES, columns=COLONNES)
    print(prudent)
    print('tableau :', 'correct' if cases == [750, 250, 90, 8910] else 'à revoir')

### b) Les deux probabilités conditionnelles

In [ ]:
# À COMPLÉTER : dans quel groupe comptez-vous ? Écrivez le quotient avec les effectifs.
p_signale_si_spam = None
p_spam_si_signale = None

In [ ]:
# Vérification
for nom, la_votre, vrai in [('P(signalé | spam)', p_signale_si_spam, 0.75),
                            ('P(spam | signalé)', p_spam_si_signale, 750 / 840)]:
    print(f'{nom} :', 'pas encore complété' if la_votre is None else
          ('correct' if abs(la_votre - vrai) < 0.001 else 'à revoir'))

### c) à e) Le théorème de Bayes en fonction

In [ ]:
# À COMPLÉTER : P(spam | signalé) à partir du taux de base p et des taux du filtre d et f
def proba_spam_si_signale(p, d, f):
    return None   # À COMPLÉTER (équations 4.4 et 4.5)

In [ ]:
# Vérification de c), d) et e)
if proba_spam_si_signale(0.1, 0.75, 0.01) is None:
    print('Complétez la fonction.')
else:
    cas = [('c) prudent, 10 % de spams', (0.10, 0.75, 0.01), 0.8929),
           ('d) prudent, 1 % de spams', (0.01, 0.75, 0.01), 0.4310),
           ('e) prudent, 2 % de spams', (0.02, 0.75, 0.01), 0.6048),
           ('e) filtre des leçons, 2 % de spams', (0.02, 0.90, 0.05), 0.2687)]
    for nom, args, attendu in cas:
        v = proba_spam_si_signale(*args)
        print(f'{nom} : {v:.4f} →', 'correct' if abs(v - attendu) < 0.0001 else 'à revoir')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, proba_spam_si_signale(P, 0.90, 0.05), color=SIGNALE, linewidth=2,
             label='filtre des leçons (90 % ; 5 %)')
    plt.plot(100 * P, proba_spam_si_signale(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='filtre prudent (75 % ; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Taux de base : part de spams (%)'); plt.ylabel('P(spam | signalé)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) et e) Vos réponses

*Votre prédiction pour d), avant de calculer :* …

*Votre choix en e) :* …

## Exercice 4.3 — Trois spams, puis dix messages

*Notions : épreuve de Bernoulli et loi binomiale (leçon 4.5), probabilité conditionnelle (leçon 4.3).* Faites d'abord les questions sur papier.

### a) à e) Sur papier

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : les probabilités de 0, 1, 2, 3 spams manqués parmi 3, puis la réponse d)
loi_3 = None        # liste de quatre nombres
p_d = None          # au moins un spam qui passe parmi 10 messages reçus

In [ ]:
# Vérification
if loi_3 is None or p_d is None:
    print('Complétez la cellule précédente.')
else:
    print('b) :', [round(v, 3) for v in loi_3], '→', 'correct' if np.allclose(loi_3, [0.729, 0.243, 0.027, 0.001]) else 'à revoir')
    print(f'd) : {p_d:.4f} →', 'correct' if abs(p_d - 0.1829) < 0.0001 else 'à revoir')

## Exercice 4.4 — Simuler dix spams

*Notions : loi binomiale (leçon 4.5), fréquence et probabilité.* On compte sur **dix spams**.

In [ ]:
# À COMPLÉTER : une série de 10 spams, chacun manqué avec la probabilité 0,1
def simuler_serie(rng):
    """Renvoie le nombre de spams manqués parmi 10. rng.random() tire un nombre entre 0 et 1."""
    return None   # À COMPLÉTER

In [ ]:
# b) et c) 10 000 séries (graine fixée pour retrouver les mêmes nombres)
rng = np.random.default_rng(4)
if simuler_serie(rng) is None:
    print('Complétez la fonction.')
else:
    rng = np.random.default_rng(4)
    X = np.array([simuler_serie(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'fréquence observée : {freq:.4f} ; probabilité calculée : {1 - 0.9 ** 10:.4f} →',
          'correct' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'à revoir')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=SIGNALE, label='fréquence observée')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probabilité (équation 4.7)')
    plt.xlabel('Spams manqués parmi 10'); plt.legend(); plt.show()
    # d) Plus de séries : la fréquence se rapproche de la probabilité
    for N in (100, 100, 100, 100000):
        Y = np.array([simuler_serie(rng) for _ in range(N)])
        print(f'{N:>6} séries : fréquence = {np.mean(Y >= 1):.4f}')

### d) Votre interprétation

*Votre réponse :* …

## Exercice 4.5 — Prédire, puis vérifier : une campagne de spams

*Notions : indépendance, événement complémentaire (leçon 4.5).*

### a) à c) Vos prédictions et calculs

*Votre prédiction :* …

In [ ]:
# À COMPLÉTER : une campagne de 10 spams identiques, manqués tous ensemble avec la probabilité 0,1
def simuler_campagne(rng):
    return None   # À COMPLÉTER : renvoie 10 ou 0

In [ ]:
# d) Vérification par simulation
rng = np.random.default_rng(5)
if simuler_campagne(rng) is None:
    print('Complétez la fonction.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([simuler_campagne(rng) for _ in range(10000)])
    print(f'au moins un manqué : {np.mean(Z >= 1):.4f} ; moyenne : {Z.mean():.3f} →',
          'correct' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'à revoir')

## Exercice 4.6 — Défi : combien de spams pour dépasser 99 % de risque d'en manquer un ?

*Notions : lois de Bernoulli et binomiale, loi normale (leçon 4.5), écart-type (leçon 1.1).*

### a) à d) Vos démonstrations et calculs

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : b) le plus petit n tel que 0,9 ** n < 0,01 ; c) moyenne et écart-type pour n = 100
n_min = None
moyenne_100 = None
ecart_type_100 = None

In [ ]:
# Vérification, puis d) binomiale contre loi normale
if None in (n_min, moyenne_100, ecart_type_100):
    print('Complétez la cellule précédente.')
else:
    print(f'n_min = {n_min} →', 'correct' if n_min == 44 else 'à revoir')
    print(f'moyenne = {moyenne_100:.1f}, écart-type = {ecart_type_100:.1f} →',
          'correct' if np.isclose(moyenne_100, 10) and np.isclose(ecart_type_100, 3) else 'à revoir')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='loi binomiale (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='loi normale (moyenne 10, écart-type 3)')
    plt.xlabel('Spams manqués parmi 100'); plt.legend(); plt.show()